In [3]:
import pandas as pd
import numpy as np
import networkx as nx
import random
import matplotlib.pyplot as plt
import datetime
from datetime import timedelta
from pulp import *
import pandas as pd
import joblib
import networkx as nx
import numpy as np
G = joblib.load('ALL_INDIA_GRAPH_STATION.pkl')
G.edges(data=True)
tt = pd.read_csv('ETA_LOADED_ENSEMBLE.csv')
tt = tt.drop(columns={'Unnamed: 0'})
tts = tt.mean(axis=1)
ids = pd.read_csv('segments_lko.csv')
ids['tt']=tts
ids['SG']=ids['SEGMENT'].str.split('-')


In [4]:
df_wagon_due_roh=joblib.load("df_wagon_due_roh.pkl")

In [5]:
df_wagon_under_maint=joblib.load("df_wagon_under_maint.pkl")
df_wagon_due_roh=df_wagon_due_roh[~df_wagon_due_roh.wagon_no.isin(df_wagon_under_maint.wagon_no)]
df_wagon_due_roh['wagon_var']=df_wagon_due_roh['wagon_no']+'_'+df_wagon_due_roh['wagon_type']+'_'+df_wagon_due_roh['ravsttn']
df_wagon_on_loc=df_wagon_due_roh.groupby(["ravsttn","wagon_type"])['wagon_no'].count()
df_wagon_on_loc=df_wagon_on_loc.reset_index()
df_wagon_curr_holding=joblib.load("df_wagon_curr_holding.pkl")
df_depo_capacity=joblib.load("df_depo_capacity.pkl")
df_wagon_type=joblib.load("df_wagon_type.pkl")
df_wagon_depo=joblib.load("df_wagon_depo.pkl")

In [6]:
df_wagon_type_comb=df_wagon_depo.merge(df_wagon_type, how='cross')
df_wagon_comb_type_ava=joblib.load("df_wagon_comb_type_ava.pkl")
df_wagon_type_comb['workdepot_wagon_type']='('+df_wagon_type_comb['depot']+','+df_wagon_type_comb['wagon_type']+')'
df_wagon_type_comb['present'] = df_wagon_type_comb['workdepot_wagon_type'].isin(df_wagon_comb_type_ava['workdepot_wagon_type'])
df_wagon_type_comb["present"] = df_wagon_type_comb["present"].astype(int)
df_wagon_type_comb=df_wagon_type_comb.reset_index(drop=True)
df_wagon_type_comb.head()
wagon_handled=df_wagon_type_comb.groupby('depot')[['wagon_type','present',]].apply(lambda x: x.set_index('wagon_type').to_dict(orient='index')).to_dict()

In [7]:
vars=LpVariable.dicts("var",(df_wagon_due_roh['wagon_var'].values,df_wagon_type_comb['depot'].unique()),cat='Binary')
prob = LpProblem("Minimize wagon time to reach workdepot", LpMinimize)
wgn_workdepot = [(w, b) for w in  df_wagon_due_roh['wagon_var'].values for b in df_wagon_type_comb['depot'].unique()]

C:\Users\Acer\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pulp\pulp.py:1352: UserWarning: Spaces are not permitted in the name. Converted to '_'
  warnings.warn("Spaces are not permitted in the name. Converted to '_'")


In [8]:
wgn_workdepot = [(w, b) for w in  df_wagon_due_roh['wagon_var'].values for b in df_wagon_type_comb['depot'].unique()]
df_final=pd.DataFrame()
for j in df_wagon_type_comb['depot'].unique():
    df=pd.DataFrame()
    wrk_shp_locn=j[:len(j)-2]
    cost=[]
    for i in df_wagon_due_roh['ravsttn'].unique():
        try:
            t=nx.shortest_path_length(G, source=i, target=wrk_shp_locn,weight='TRAVELTIME')
            cost.append(t)
        except:
            if(len(cost)==0):
                cost.append(0)
            else:
                cost.append(max(cost))
    df['ravsttn']=df_wagon_due_roh['ravsttn'].unique()
    df['cost']=pd.DataFrame(cost)
    df['workdepot']=j
    df_final=pd.concat(([df_final,df]))

In [9]:
df_final.head()

,ravsttn,cost,workdepot
0,FN,21,ADTPFD
1,MBPJ,35,ADTPFD
2,GOSG,35,ADTPFD
3,CSCP,12,ADTPFD
4,BSCS,12,ADTPFD


In [10]:
df_cost=pd.merge(df_wagon_due_roh,df_final,how='inner')
df_cost=df_cost[['wagon_var','workdepot','cost']]
df_cost['cost'].replace(to_replace = 0, value = df_cost['cost'].mean(), inplace=True)
df = df_cost.groupby('wagon_var')[['workdepot','cost']].apply(lambda x: x.set_index('workdepot').to_dict(orient='index')).to_dict()
costs=df

C:\Users\Acer\AppData\Local\Temp\ipykernel_7080\3789995790.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_cost['cost'].replace(to_replace = 0, value = df_cost['cost'].mean(), inplace=True)


In [11]:
prob += (
    lpSum([vars[w][b] * costs[w][b]['cost'] for (w, b) in wgn_workdepot]),
    "Sum_of_wagon_Transporting_Costs",)
Workdepot_capcity=df_depo_capacity.set_index('depot').to_dict('dict')
Workdepot_capcity=Workdepot_capcity['depot_capacity']
Current_wagon_load_wordepot=df_wagon_curr_holding.set_index('depot').to_dict()
Current_wagon_load_wordepot=Current_wagon_load_wordepot['current_holding']
for w in df_wagon_due_roh['wagon_var'].values:
    for b in df_wagon_type_comb['depot'].unique():
        prob+= (
            (vars[w][b]) <= 1,
            "Variable_can_be_either_zero_or_one"+str(vars[w][b])+"%s" % w,
    )

In [12]:
for b in df_wagon_type_comb['depot'].unique():
    try:
        prob += (
            lpSum([vars[w][b]  for w in df_wagon_due_roh['wagon_var'].values]) == max(Workdepot_capcity[b]-Current_wagon_load_wordepot[b],0),
            "Sum_of_wagons_assigned_should_be_less_than_capacity"+"%s" % b,)   
    except:
        prob += (
            lpSum([vars[w][b]  for w in df_wagon_due_roh['wagon_var'].values]) == max(Workdepot_capcity[b]-0,0),
            "Sum_of_wagons_assigned_should_be_less_than_capacity"+"%s" % b,) 

In [13]:
for w in df_wagon_due_roh['wagon_var'].values:
    df_wagon_type=df_wagon_due_roh[df_wagon_due_roh['wagon_var']==w]
    df_wagon_type=df_wagon_type.reset_index(drop=True)
    type=df_wagon_type['wagon_type'][0]
    for b in df_wagon_type_comb['depot'].unique():
        try:
            handled=wagon_handled[b][type]['present']
        except:
            handled=0
        try:
            prob+= (
            (vars[w][b]) <= handled,
            "wagon_handled_current_either_zero_or_one"+str(b)+"%s" % w,
    )
        except:
            prob+= (
            (vars[w][b]) <= handled,
            "wagon_handled_current_either_zero_or_one"+str(b)+"%s" % w,
    )

In [14]:
wagon_locn_dict=df_wagon_on_loc.groupby('ravsttn')[['wagon_type','wagon_no',]].apply(lambda x: x.set_index('wagon_type').to_dict(orient='index')).to_dict()
wagon_locn_dict   
location=df_wagon_on_loc['ravsttn'].unique()
for c in location:
    df_wagon_type=df_wagon_type[df_wagon_type['ravsttn']==c]
    for type in df_wagon_type['wagon_type'].unique():
        try:
            prob += (
            lpSum([vars[w][b]  for w in df_wagon_due_roh['wagon_var'].values if type in w if c in w for b in df_wagon_type_comb['depot'].unique()]) <= wagon_locn_dict[c][type]['wagon_no'],
            "Sum_of_wagons_assigned_should_be_less_than_capacity"+"%s" % c+type,
        )
        except:
            continue

In [15]:
print(prob.solve(),LpStatus[prob.status])

1 Optimal


In [16]:
counter=0
var=[]
for v in prob.variables():
      if (v.varValue!=0):
        counter=counter+1
        var.append(v.name)

In [17]:
vars_list_final=[]
for k in range(len(var)):
    vars_list=var[k].split('_')
    vars_list_final.append(vars_list)
df_result = pd.DataFrame(vars_list_final, columns =['var', 'wagon_number','wagon_type','Current_wagon_location','workdepot']) 
df_result.head()

,var,wagon_number,wagon_type,Current_wagon_location,workdepot
0,var,10010101703,BOXNM1,MBPJ,BMYREPFD
1,var,10039672635,BOXNM1,OTPS,OBRFD
2,var,10068960260,MBOXN,BNE,ETFD
3,var,10070660202,BOXNM1,TPT,JTJFD
4,var,10079250497,BOXNM1,CLX,NLPDFD


In [18]:
df_cost[['wagon_number','wagon_type','Current_wagon_location']] = df_cost['wagon_var'].str.split('_',expand=True)
df_cost.head()

,wagon_var,workdepot,cost,wagon_number,wagon_type,Current_wagon_location
0,00506_BTAP_FN,ADTPFD,21.000000,00506,BTAP,FN
1,00506_BTAP_FN,AQICDFD,23.171314,00506,BTAP,FN
2,00506_BTAP_FN,BADFD,10.000000,00506,BTAP,FN
3,00506_BTAP_FN,BIAEXYDFD,23.171314,00506,BTAP,FN
4,00506_BTAP_FN,BIAPPYDFD,23.171314,00506,BTAP,FN


In [19]:
df_matrix=pd.merge(df_result,df_cost,how='inner',on=['wagon_number','wagon_type','Current_wagon_location','workdepot'])

In [20]:
df_matrix

,var,wagon_number,wagon_type,Current_wagon_location,workdepot,wagon_var,cost
0,var,10010101703,BOXNM1,MBPJ,BMYREPFD,10010101703_BOXNM1_MBPJ,23.171314
1,var,10039672635,BOXNM1,OTPS,OBRFD,10039672635_BOXNM1_OTPS,1.000000
2,var,10068960260,MBOXN,BNE,ETFD,10068960260_MBOXN_BNE,3.000000
3,var,10070660202,BOXNM1,TPT,JTJFD,10070660202_BOXNM1_TPT,1.000000
4,var,10079250497,BOXNM1,CLX,NLPDFD,10079250497_BOXNM1_CLX,4.000000
5,var,10079666021,BOXNM1,MSCA,BPAFD,10079666021_BOXNM1_MSCA,2.000000
6,var,10079768985,BOXNM1,TPT,JTJFD,10079768985_BOXNM1_TPT,1.000000
7,var,10079961959,BOXNM1,SCRM,BPAFD,10079961959_BOXNM1_SCRM,2.000000
8,var,10080160037,BOXNM1,MSCA,BPAFD,10080160037_BOXNM1_MSCA,2.000000
9,var,10089700111,BOXNM1,OTPS,OBRFD,10089700111_BOXNM1_OTPS,1.000000


In [21]:
sum_1=0
for i in df_wagon_type_comb['depot'].unique():
    try:
        sum_1=sum_1+max(Workdepot_capcity[i]-Current_wagon_load_wordepot[i],0)
    except:
        sum_1=sum_1+max(Workdepot_capcity[i]-0,0)
print(sum_1)

60


In [22]:
sum_2=0
for j in Current_wagon_load_wordepot:
    sum_2=sum_2+Current_wagon_load_wordepot[j]
print(sum_2)


865


In [23]:
sum_3=0
for j in Workdepot_capcity:
    sum_3=sum_3+Workdepot_capcity[j]
print(sum_3)

392
